<a href="https://colab.research.google.com/github/MottaDavide/MaskArchitectureAnomaly_CourseProject/blob/finetuning%2Fcoco-to-cityscapes/Task5_fineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fine-tuning COCO → Cityscapes
# Head-only fine-tuning: freeze everything except `class_head`; full validation + WandB + resumable checkpoints

#Setting the environment
1. Mounting Google Drive
2. Cloning from GitHub
3. Installing requirements

In [ ]:
import os
import shutil
from google.colab import drive
# !pip install ood_metrics --no-deps

# Monta Google Drive
drive.mount("/content/drive")

# ================= CONFIGURAZIONE =================
DRIVE_ROOT = "/content/drive/MyDrive"

# Cartella su Drive dove salvare/clonare la repo
PROJECT_FOLDER = "MaskArchitectureAnomaly_CourseProject"

# Repo GitHub pubblica
GIT_USERNAME = "ChiaraApolito"
REPO_NAME = "MaskArchitectureAnomaly_CourseProject"

# Branch da usare
BRANCH_NAME = "finetuning/coco-to-cityscapes"
# oppure "main" se vuoi il main

project_path = os.path.join(DRIVE_ROOT, PROJECT_FOLDER)

repo_url = f"https://github.com/{GIT_USERNAME}/{REPO_NAME}.git"
# ==================================================


def setup_repository():
    print(f"\n--- Gestione progetto: {PROJECT_FOLDER} ---")

    if not os.path.exists(project_path):
        print("📂 Clonazione repository...")
        os.system(
            f'git clone --branch "{BRANCH_NAME}" "{repo_url}" "{project_path}"'
        )

    else:
        git_dir = os.path.join(project_path, ".git")

        if not os.path.exists(git_dir):
            print("⚠️ Cartella esistente ma non è una repo Git. La ricreo...")
            shutil.rmtree(project_path)
            os.system(
                f'git clone --branch "{BRANCH_NAME}" "{repo_url}" "{project_path}"'
            )

        else:
            print("🔄 Aggiornamento repository...")
            os.chdir(project_path)

            # Siccome non devi fare modifiche su Colab, meglio eliminare eventuali cambi locali
            os.system("git fetch origin")
            os.system(f"git checkout {BRANCH_NAME}")
            os.system(f"git reset --hard origin/{BRANCH_NAME}")

    print("✅ Repository pronta.")
    print("Percorso progetto:", project_path)


# Setup Repo
setup_repository()

# Esecuzione Script
os.chdir(project_path)

# installa dipendenze della repo
get_ipython().system("pip install -r eomt/requirements.txt")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Login in wandb
Copy your api key (you can leave this one to save data to the same account for this project)

In [6]:
import os
from getpass import getpass

# Non salvare la API key in chiaro nel notebook.
# Inseriscila quando Colab la chiede, oppure imposta WANDB_API_KEY nei Secrets di Colab.
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass("WandB API key: ")

import wandb
wandb.login(key=os.environ["WANDB_API_KEY"])


# Patch/controllo codice per head-only fine-tuning
Questa cella aggiunge `freeze_all_except_class_head` al LightningModule se il branch clonato non lo contiene già. In questo modo il config può congelare tutto il modello EoMT tranne `network.class_head`.

In [ ]:
# from pathlib import Path
# import zipfile
# import re

# PROJECT_ROOT = Path.cwd()
# lm_path = PROJECT_ROOT / "eomt" / "training" / "lightning_module.py"
# assert lm_path.exists(), f"File non trovato: {lm_path}"

# text = lm_path.read_text()

# if "freeze_all_except_class_head" not in text:
#     # 1) Aggiunge il parametro alla firma del __init__ subito dopo freeze_encoder_except_last_n
#     text = text.replace(
#         "        freeze_encoder: bool = False,\n        freeze_encoder_except_last_n: int = 0,\n    ):",
#         "        freeze_encoder: bool = False,\n        freeze_encoder_except_last_n: int = 0,\n        freeze_all_except_class_head: bool = False,\n    ):",
#     )

#     # 2) Sostituisce il blocco freeze encoder con un blocco a priorità head-only
#     old = """        if freeze_encoder_except_last_n > 0:\n            blocks = list(self.network.encoder.backbone.blocks)\n            for block in blocks[:-freeze_encoder_except_last_n]:\n                for param in block.parameters():\n                    param.requires_grad_(False)\n        elif freeze_encoder:\n            for param in self.network.encoder.parameters():\n                param.requires_grad_(False)\n"""

#     new = """        if freeze_all_except_class_head:\n            # Freeze the whole EoMT network\n            for param in self.network.parameters():\n                param.requires_grad_(False)\n\n            # Unfreeze only the classification head\n            if not hasattr(self.network, \"class_head\"):\n                raise AttributeError(\n                    \"freeze_all_except_class_head=True requires self.network.class_head\"\n                )\n            for param in self.network.class_head.parameters():\n                param.requires_grad_(True)\n\n            trainable = [\n                name for name, param in self.network.named_parameters()\n                if param.requires_grad\n            ]\n            logging.info(\"Trainable network parameters after head-only freeze:\")\n            for name in trainable:\n                logging.info(f\"  {name}\")\n\n            if len(trainable) == 0:\n                raise RuntimeError(\"No trainable parameters found in class_head.\")\n\n        elif freeze_encoder_except_last_n > 0:\n            blocks = list(self.network.encoder.backbone.blocks)\n            for block in blocks[:-freeze_encoder_except_last_n]:\n                for param in block.parameters():\n                    param.requires_grad_(False)\n        elif freeze_encoder:\n            for param in self.network.encoder.parameters():\n                param.requires_grad_(False)\n"""

#     if old not in text:
#         raise RuntimeError("Blocco freeze originale non trovato: controlla manualmente lightning_module.py")

#     text = text.replace(old, new)
#     lm_path.write_text(text)
#     print("Patch applicata: freeze_all_except_class_head aggiunto.")
# else:
#     print("freeze_all_except_class_head già presente nel codice.")

# # Verifica rapida
# patched = lm_path.read_text()
# assert "freeze_all_except_class_head: bool = False" in patched
# assert "self.network.class_head.parameters()" in patched
# print("Controllo OK.")


# Fine-tuning instructions aggiornate
1. Per partire dal modello COCO pretrained usa la cella **START FROM COCO**.
2. Per riprendere un training interrotto usa la cella **RESUME FROM LAST CHECKPOINT**.
3. La validazione non è più disattivata: viene eseguita interamente e loggata su WandB.
4. I checkpoint vengono salvati su Google Drive sia periodicamente per step sia come `last.ckpt`.

In [ ]:
# RESUME FROM LAST CHECKPOINT
# Riprende modello + optimizer + scheduler dal checkpoint Lightning salvato su Drive.
# Mantieni --model.init_args.ckpt_path al COCO: serve per costruire il modello,
# poi --ckpt_path sovrascrive lo stato con il checkpoint di training.

%cd /content/MaskArchitectureAnomaly_CourseProject/eomt

CKPT_DIR = "/content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_head_only"
LAST_CKPT = f"{CKPT_DIR}/last.ckpt"

!python main.py fit \
  -c configs/dinov2/finetuning/coco_to_cityscapes_freeze_backbone.yaml \
  --trainer.devices 1 \
  --trainer.max_epochs 50 \
  --trainer.precision 16-mixed \
  --trainer.log_every_n_steps 1 \
  --trainer.check_val_every_n_epoch 1 \
  --trainer.limit_val_batches 1.0 \
  --data.batch_size 1 \
  --data.path /content \
  --model.init_args.ckpt_path /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/eomt_coco.bin \
  --model.init_args.load_ckpt_class_head false \
  --model.init_args.freeze_all_except_class_head true \
  --trainer.default_root_dir $CKPT_DIR \
  --trainer.logger.init_args.project eomt \
  --trainer.logger.init_args.name coco_to_cityscapes_head_only \
  --trainer.logger.init_args.resume allow \
  --trainer.callbacks+='{"class_path":"lightning.pytorch.callbacks.ModelCheckpoint","init_args":{"dirpath":"/content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_head_only","filename":"epoch={epoch}-step={step}","save_top_k":-1,"monitor":null,"every_n_train_steps":1000,"save_last":true}}' \
  --ckpt_path $LAST_CKPT


/content/MaskArchitectureAnomaly_CourseProject/eomt
2026-05-02 09:11:52.279954: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Interpolated pos_embed from 1600 to 4096 tokens
INFO:root:Loaded 195 keys
Using 16bit Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: Currently logged in as: miriage96 (miriage96-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.19.10
wandb: Run data is s

In [7]:
# START FROM COCO PRETRAINED - HEAD ONLY
# Avvia il fine-tuning da eomt_coco.bin, non carica la vecchia head COCO,
# congela tutto il modello e lascia allenabile solo network.class_head.

%cd /content/MaskArchitectureAnomaly_CourseProject/eomt

CKPT_DIR = "/content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_head_only"

!python main.py fit \
  -c configs/dinov2/finetuning/coco_to_cityscapes_freeze_backbone.yaml \
  --trainer.devices 1 \
  --trainer.max_epochs 50 \
  --trainer.precision 16-mixed \
  --trainer.log_every_n_steps 1 \
  --trainer.check_val_every_n_epoch 1 \
  --trainer.limit_val_batches 1.0 \
  --data.batch_size 1 \
  --data.path /content \
  --model.init_args.ckpt_path /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/eomt_coco.bin \
  --model.init_args.load_ckpt_class_head false \
  --model.init_args.freeze_all_except_class_head true \
  --trainer.default_root_dir $CKPT_DIR \
  --trainer.logger.init_args.project eomt \
  --trainer.logger.init_args.name coco_to_cityscapes_head_only \
  --trainer.logger.init_args.resume allow \
  --trainer.callbacks+='{"class_path":"lightning.pytorch.callbacks.ModelCheckpoint","init_args":{"dirpath":"/content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_head_only","filename":"epoch={epoch}-step={step}","save_top_k":-1,"monitor":null,"every_n_train_steps":1000,"save_last":true}}'


/content/MaskArchitectureAnomaly_CourseProject/eomt
2026-05-02 08:54:33.479925: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
usage: main.py [options] fit [-h] [-c CONFIG] [--print_config [=flags]]
                             [--seed_everything SEED_EVERYTHING]
                             [--trainer CONFIG]
                             [--trainer.accelerator.help CLASS_PATH_OR_NAME]
                             [--trainer.accelerator ACCELERATOR]
                             [--trainer.strategy.help CLASS_PATH_OR_NAME]
                             [--trainer.strategy STRATEGY]
                             [--trainer.devices DEVICES]
                             [--trainer.num_nodes NUM_NODES]
                             [--trainer.precision